# Reproducing Asari & Zador (2009) on CRCNS ac-1

Paper: H. Asari and A. M. Zador (2009), "Long-Lasting Context Dependence Constrains Neural
Encoding Models in Rodent Auditory Cortex", J Neurophysiol 102:2638-2656, doi:10.1152/jn.00577.2009.
The claims file (`claims/asari-2009.yaml`) was written from the version of record (a text
extraction of the publisher's PDF); the online supplement was not read.

Dataset: CRCNS ac-1 (doi:10.6080/K0KW5CXR), read in place from the public S3 bucket
`crcnsarchive` (`ac-1/crcns-ac1.zip`, `crcns-ac1-asari-results-1.zip`,
`crcns-ac1-asari-results-2.zip`). CRCNS data sets are not versioned, so the version is
`unversioned`; it is pinned by the MD5 sums of the three zips, printed below from the data set's
`checksums.md5` (docs/archives.md, "CRCNS"). This is the only published version, so it is the one
used. Of the data set, this notebook uses Hiroki Asari's recordings (`asari/` and the two results
zips). Michael Wehr's recordings (`wehr/`), which the paper reanalyses for its STRF models
(Figure 4), are not used: that claim was not run in the time of this attempt.

This is a repeat attempt, made blind to earlier attempts.

| Claim | Tested here |
|---|---|
| methods-recording-counts | tested (voltage-clamp counts: not in the data set) |
| a1-mgb-firing-rates | tested |
| fig2-example-reliability | tested |
| fig2-example-long-context | tested |
| fig3a-proportion-significant | tested |
| fig3a-significance-decay | tested |
| fig3b-fractional-power-decay | tested |
| timescale-longer-than-membrane | tested |
| fig4-predictability-upper-bound | tested |
| fig4-strf-performance-low | blocked: not run within the time of this attempt |
| fig5-example-conductances | blocked: no voltage-clamp recordings in the data set |
| fig6-conductance-decay | blocked: no voltage-clamp recordings in the data set |
| fig8a-proportion-by-property | tested |
| fig8b-decay-by-property | tested |
| fig8a-significance-fits | tested |
| fig8c-relative-effects | tested (probe resampling only) |
| fig8-natural-synthetic-probes-alike | tested |
| fig9-mgb-fast-decay | tested |
| fig9-example-first-bin | tested |
| fig7-example-frequency-context | blocked: no recording matches the example |
| figs1-silence-probe | tested |

Method details the claims file leaves to the paper are taken from the Methods, and where the
Methods leave them open, from the authors' analysis code that ships inside the data set
(`asari/Tools/CxtDep.m` and its helpers). The helper module `asari.py` says, for each parameter,
where it comes from.

In [ ]:
import os, time, zlib, collections, resource
os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
from scipy import stats
import asari as A

PAPER = "asari-2009"
def seed_of(claim):
    return zlib.crc32(f"{PAPER}/{claim}".encode())
def five(claim):
    s = seed_of(claim)
    return [s, s + 1000, s + 2000, s + 3000, s + 4000]
np.set_printoptions(precision=4, suppress=True)
T0 = time.time()

## Data

The data set's checksums and its own documentation are read first. `asari/Summary.mat` lists the
experiments and their files; the tuning-curve files are not used.

In [ ]:
import fsspec
md5 = fsspec.open(A.BUCKET + "checksums.md5", "rt").open().read()
print(md5)
for name, h in A.MD5.items():
    assert h in md5, name
print(A.read_text("asari/Contents.m")[:1500])
F = A.file_lists()
print({k: len(v) for k, v in F.items()})
print("files listed in Summary.mat but absent from the zips:", F["absent"])

Each file is read once, in four processes, and reduced to what the claims need
(`A.reduce_wholecell`, `A.reduce_mgb`): for every probe its significant periods, the shuffle noise
floor (1000 shuffles under each of five seeds), the power estimates, trial correlations and
autocorrelations, at 1-ms resolution; spike counts for the rates.

In [ ]:
t = time.time()
SEEDS_NAT = five("fig3a-proportion-significant")
SEEDS_SYN = five("fig8a-significance-fits")
NAT = A.run_many(A.reduce_wholecell, [(p, "natural", SEEDS_NAT, i) for i, p in enumerate(F["natural"])])
print("natural files", len(NAT), round(time.time() - t), "s")
t = time.time()
MGB = A.run_many(A.reduce_mgb, [(p,) for p in F["mgb"]])
print("MGB files", len(MGB), round(time.time() - t), "s")
t = time.time()
SYN = A.run_many(A.reduce_wholecell, [(p, "synthetic", SEEDS_SYN, i) for i, p in enumerate(F["synthetic"])])
print("synthetic files", len(SYN), round(time.time() - t), "s")

In [ ]:
# Data check: counts, ranges and units of every variable used.
def cells_of(R): return sorted({r["cell"] for r in R})
for name, R in (("A1 natural", NAT), ("A1 synthetic", SYN), ("MGB", MGB)):
    an = sorted({r["animal"] for r in R})
    print(f"{name}: {len(R)} files, {len(cells_of(R))} cells, {len(an)} animal-days; "
          f"effective repeats {sorted(collections.Counter(r['rep'] for r in R).items())}")
units = collections.Counter(r["unit"] for r in NAT + SYN)
sfs = collections.Counter(r["sf"] for r in NAT + SYN)
print("units (current clamp):", units, " sampling rates:", sfs)
vr = np.array([r["vrange"] for r in NAT + SYN])
print("median-filtered membrane potential, 1st and 99th percentiles over files: "
      f"{vr[:, 0].min():.1f} to {vr[:, 1].max():.1f} mV (median of file medians of the 1st pct {np.median(vr[:, 0]):.1f})")
ages = collections.Counter(r["age"] for r in NAT + SYN)
print("ages (days):", sorted(ages.items()))
print("fragment lengths (s), natural:", round(min(min(r['frag_s']) for r in NAT), 2), "to",
      round(max(max(r['frag_s']) for r in NAT), 2))
print("recording time per natural file: silence", round(np.median([r['silence_s'] for r in NAT])),
      "s, sound", round(np.median([r['sound_s'] for r in NAT])), "s (medians)")
# Voltage clamp: every recording is current clamp ('Whole-cell', mV) or cell-attached.
sm = A.summary()
print("experiments in Summary.mat:", sorted({(r['experiment'], r['experiment_description']) for r in sm}))
# Recordings deposited twice: identical probe responses would give identical power traces.
sig = collections.Counter()
for r in NAT + SYN:
    for t in r["tests"][:1]:
        sig[(round(float(t['pred'][:50].sum()), 6), t['T'])] += 1
print("files sharing their first probe's power trace (duplicates):", sum(v > 1 for v in sig.values()))
print("peak memory so far (MB):", resource.getrusage(resource.RUSAGE_SELF).ru_maxrss // 1024)

Clock check. The trial tables give the onset of each sequence in samples. If they share the
trace's clock, responses to repeats of a probe in one context are correlated when aligned at
those onsets and much less when each trial is moved by an independent random 0.5 to 1.5 s; in
the thalamus, the rate in the first 100 ms after a probe onset exceeds the rate in the next 900 ms
only when aligned. Two files of each kind are checked.

In [ ]:
chk = []
for p in F["natural"][:1] + F["natural"][-1:]:
    chk.append(("cc", p, A.alignment_check(p, "cc", seed_of("methods-recording-counts"))))
for p in F["mgb"][:1] + F["mgb"][-1:]:
    chk.append(("ca", p, A.alignment_check(p, "ca", seed_of("methods-recording-counts"))))
for k, p, r in chk:
    what = "trial-to-trial correlation, first 1 s" if k == "cc" else "onset rate excess (spike/s)"
    print(f"{p.split('/')[-1]}: {what}: aligned {r['true']:.3f}, jittered {r['jitter']:.3f} ({r['n']} probe-contexts)")

## methods-recording-counts

> Of 194 A1 cells recorded in current clamp, 123 met the analysis criterion (39 tested with
> natural sounds, 14 with TORCs, 6 with DMRs, 39 with modulated harmonic tones, 27 with modulated
> colored noise); 14 of 42 voltage-clamped cells and 14 of 16 MGB cells were analysed.

A probe counts when it has at least two conditioning stimuli with at least 4 trials each (current
clamp; the file's "Effective repeats", CxtDep.m) or at least 10 (MGB, Methods, Context dependence
at suprathreshold level). Both readings of silence (counted as a conditioning stimulus or not) are
shown. For the synthetic ensembles, Table 1's columns are formed by `A.synthetic_tests` (each
probe with each set of conditioning stimuli that differ in one property).

In [ ]:
def probes(R, minrep, sub=None):
    out = []
    for r in R:
        if (r["rep"] or 0) < minrep:
            continue
        for t in r["tests"]:
            if sub == "ns" and "ns" not in t:
                continue
            out.append((r, t))
    return out
nat = probes(NAT, 4)
nat_ns = probes(NAT, 4, "ns")
print("A1 natural sound probes (silence counted):", len(nat), "in", len({r['cell'] for r, _ in nat}), "cells")
print("A1 natural sound probes (silence not counted):", len(nat_ns), "in", len({r['cell'] for r, _ in nat_ns}), "cells")
print("natural sound files with fewer than 4 effective repeats:", [r['path'] for r in NAT if (r['rep'] or 0) < 4])
mgb_pr = [(r, p) for r in MGB if (r["rep"] or 0) >= 10 for p in r["probes_code"]]
print("MGB probes (>= 10 trials):", len(mgb_pr), "in", len({r['cell'] for r, _ in mgb_pr}), "cells;",
      "MGB cells in the data set:", len(cells_of(MGB)))
syn = probes(SYN, 4)
def typ(r):
    d = r["description"]
    return d
row_of = {}
tab = collections.Counter(); tabc = collections.defaultdict(set)
for r, t in syn:
    stim = r["stim"][t["probe"] - 1]
    row = ("NS" if t["natural"] else "TORC" if "TORC" in stim else "DMR" if "DMR" in stim
           else "MHT" if "Modulated" in stim else "MCN" if "pink" in stim else "other")
    tab[(row, t["prop"])] += 1; tabc[(row, t["prop"])].add(r["cell"])
print("\nTable 1 as formed here: probes (cells) by probe type and property")
for row in ("NS", "TORC", "DMR", "MHT", "MCN"):
    print(f"  {row:5s}", "  ".join(f"{p}: {tab[(row, p)]} ({len(tabc[(row, p)])})" for p in A.PROPERTIES if tab[(row, p)]))
tot = collections.Counter(); totc = collections.defaultdict(set)
for r, t in syn:
    tot[t["prop"]] += 1; totc[t["prop"]].add(r["cell"])
print("  totals:", {p: (tot[p], len(totc[p])) for p in A.PROPERTIES}, " paper: amplitude 95 (31), frequency 110 (35), AM 96 (27), FM 82 (25), higher-order 137 (27)")
syn_cells = {r["cell"] for r, _ in syn}
cc_cells = syn_cells | {r["cell"] for r, _ in nat}
print("\nA1 current-clamp cells with at least one probe meeting the criterion:", len(cc_cells),
      f"(natural {len({r['cell'] for r, _ in nat})}, synthetic {len(syn_cells)}, both {len(syn_cells & {r['cell'] for r, _ in nat})})")
ens = collections.defaultdict(set)
for r, t in syn:
    s = r["stim"][t["probe"] - 1] if False else " ".join(r["stim"])
    e = ("TORC" if "TORC" in s else "DMR" if "DMR" in s else "MHT" if "Modulated" in s else "MCN" if "pink" in s else "natural variants")
    ens[e].add(r["cell"])
print("synthetic cells by ensemble:", {k: len(v) for k, v in ens.items()}, " paper: TORC 14, DMR 6, MHT 39, MCN 27")
print("all A1 current-clamp cells in the data set (Summary.mat, natural + synthetic + other):",
      len({r['cell'] for r in sm if r['experiment'] in (0, 2, 3)}))

**Result: partially reproduced.** Whether silence counts or not, 305 natural sound probes in 39
cells meet the criterion, and 93 MGB probes in 14 cells (every MGB cell of the data set), as in
Table 1. Summary.mat lists four experiments and none is in voltage clamp, so the 14 cells and 29
probes of the voltage-clamp claims cannot be counted. 133 current-clamp cells have at least one
probe meeting the criterion, not 123 (natural 39, synthetic 96, 2 in both). The property columns
formed here total 104, 120, 96, 82 and 354 tests against the paper's 95, 110, 96, 82 and 137: AM,
FM and the DMR cells match, and the higher-order column differs most, as each probe of the
natural-and-noise ensembles is tested once with each natural-noise pair.

## a1-mgb-firing-rates

> Firing rates were low in A1 (spontaneous 0.47 +/- 0.61 Hz, evoked 0.57 +/- 0.77 Hz, mean +/- SD,
> 194 cells) and high in MGB (spontaneous 0.78 +/- 1.25 Hz, evoked 11.4 +/- 16.9 Hz, 16 cells).

Spontaneous: the intersequence silences, whole or without their first second. Evoked: all sound
periods (the probe periods are the same samples, as every fragment is a probe). A1 spikes, an
assumption since the authors extracted whole-cell spikes by hand: peaks of the raw trace above
its 10-ms median by 10, 15 or 20 mV. MGB spikes: the authors' automatic detector (GetSpikes.m) and
the same with half its threshold. Rates per cell pooled over its files, then mean and SD over
cells. A1 uses the natural and synthetic context files (the tuning-curve files are not read).

In [ ]:
def rates(R, key, late=False):
    acc = collections.defaultdict(lambda: np.zeros(4))
    for r in R:
        x = r[key]
        acc[r["cell"]] += [x["silence_late" if late else "silence"], r["silence_late_s" if late else "silence_s"],
                           x["sound"], r["sound_s"]]
    sp = np.array([v[0] / v[1] for v in acc.values()]); ev = np.array([v[2] / v[3] for v in acc.values()])
    return len(acc), sp, ev
RATES = {}
print("reading                          cells  spont mean (SD)   evoked mean (SD)")
for h in (10, 15, 20):
    for late in (False, True):
        n, sp, ev = rates(NAT + SYN, f"spk{h}", late)
        RATES[("A1", h, late)] = (sp.mean(), ev.mean())
        print(f"A1 {h} mV, silence {'after 1 s' if late else 'whole  '}      {n:4d}   {sp.mean():.3f} ({sp.std(ddof=1):.3f})    {ev.mean():.3f} ({ev.std(ddof=1):.3f})")
for key in ("code", "half"):
    for late in (False, True):
        n, sp, ev = rates(MGB, f"spk_{key}", late)
        RATES[("MGB", key, late)] = (sp.mean(), ev.mean())
        print(f"MGB detector {key:4s}, silence {'after 1 s' if late else 'whole  '} {n:4d}   {sp.mean():.3f} ({sp.std(ddof=1):.3f})    {ev.mean():.3f} ({ev.std(ddof=1):.3f})")
        if key == "code" and not late:
            print("   per MGB cell evoked rates (Hz):", np.round(np.sort(ev), 2))
exp = dict(a1s=(0.47, 0.047), a1e=(0.57, 0.057), ms=(0.78, 0.078), me=(11.4, 1.14))
ok = lambda v, e: abs(v - e[0]) <= e[1]
for h in (10, 15, 20):
    for key in ("code", "half"):
        for late in (False, True):
            a = RATES[("A1", h, late)]; m = RATES[("MGB", key, late)]
            print(f"A1 {h} mV / MGB {key} / silence {'late' if late else 'whole'}: within tolerance "
                  f"A1 spont {ok(a[0], exp['a1s'])}, A1 evoked {ok(a[1], exp['a1e'])}, MGB spont {ok(m[0], exp['ms'])}, "
                  f"MGB evoked {ok(m[1], exp['me'])}; A1 < 1 Hz {max(a) < 1}, MGB evoked > 5 Hz {m[1] > 5}")

**Result: partially reproduced.** With the 15-mV detector the A1 means are 0.481 Hz spontaneous and
0.547 Hz evoked over 133 cells (SD 0.597 and 0.677), within 10% of 0.47 and 0.57, and the A1 means
are below 1 Hz under every reading. The MGB evoked mean is 4.99 Hz with the authors' detector and
7.07 Hz with half its threshold, below 11.4 Hz (tolerance 1.14) under both, and above 5 Hz only
with the lower threshold, under which the MGB spontaneous mean (0.768 Hz) is within tolerance. One
MGB cell fires at 43 Hz and nine below 1.5 Hz. The data set holds 14 MGB cells; the paper's rates
are over 16.

## fig2-example-reliability

> The example A1 neuron of Figure 2 had high trial-to-trial reliability (correlation across trials
> 0.74 +/- 0.08 for the shown probe and 0.61 +/- 0.07 over the seven natural sound fragments
> tested), and the mean correlation over the population was 0.31 +/- 0.09.

Correlation between the median-filtered traces of trials within one context, over the probe.
Readings: the mean over pairs of trials, or each trial with the mean of the others; the population
mean over probes or over cells (each cell the mean of its probes).

In [ ]:
for key in ("corr_pair", "corr_loo"):
    per_probe = np.array([t[key] for _, t in nat])
    cell = collections.defaultdict(list)
    for r, t in nat:
        cell[r["cell"]].append(t[key])
    per_cell = np.array([np.mean(v) for v in cell.values()])
    print(f"{key}: over probes {per_probe.mean():.3f} +/- {per_probe.std(ddof=1):.3f} (n={len(per_probe)}); "
          f"over cells {per_cell.mean():.3f} +/- {per_cell.std(ddof=1):.3f} (n={len(per_cell)}); "
          f"range over probes {per_probe.min():.2f} to {per_probe.max():.2f}")
# Example cell: seven natural sound fragments (the 7-long-stimuli design), per-fragment mean 0.61 +/- 0.07.
cand = []
for r in NAT:
    if len(r["tests"]) == 7 and (r["rep"] or 0) >= 4:
        v = np.array([t["corr_pair"] for t in r["tests"]])
        best = max(r["tests"], key=lambda t: -abs(t["corr_pair"] - 0.74))
        cand.append((abs(v.mean() - 0.61) + abs(best["corr_pair"] - 0.74), r["path"], v.mean(), v.std(ddof=1),
                     best["probe"], best["corr_pair"], best["corr_pair_sd"], best["contexts"], r["rep"]))
cand.sort()
print("\ncells tested with seven fragments, closest to the example first:")
for c in cand[:8]:
    print(f"  {c[1].split('/')[-1]}: fragments {c[2]:.3f} +/- {c[3]:.3f}; probe {c[4]} r = {c[5]:.3f} +/- {c[6]:.3f}, contexts {c[7]}, repeats {c[8]}")
match = [c for c in cand if abs(c[2] - 0.61) <= 0.061 and abs(c[5] - 0.74) <= 0.074]
print("cells matching both values within 10%:", [c[1].split('/')[-1] for c in match])

**Result: reproduced.** The mean pairwise correlation is 0.310 +/- 0.092 over the 39 cells and
0.319 +/- 0.129 over the 305 probes, both within 0.31 +/- 0.09; correlating each trial with the
mean of the others gives 0.437 and 0.448, outside. One cell (20051222-ha1-001) matches the example:
0.611 +/- 0.081 over its seven fragments and 0.737 +/- 0.074 for probe 4, tested after silence and
two fragments with 6 repeats.

## fig2-example-long-context

> In an example A1 neuron, changing the conditioning stimulus changed the response to the probe for
> about 4 s; responses did not differ between 2 and 4 s after probe onset and diverged again after
> about 4 s.

The example is the probe of the cell matched above (tested after silence and two fragments). The
significant periods of the matched probe, and of every three-context probe of the candidate
cells, are printed; as a weaker check, the natural sound probes with a significant period after 4 s.

In [ ]:
def pattern(t):
    iv = [(a / 1e4, b / 1e4) for a, b in t["sig"]]
    early = any(a < 2.0 for a, b in iv)
    gap = not any((b > 2.2) and (a < 3.6) for a, b in iv)
    late = any(a >= 3.6 for a, b in iv)
    return iv, early, gap, late
ex = []
for c in cand:
    r = next(r for r in NAT if r["path"] == c[1])
    for t in r["tests"]:
        if len(t["contexts"]) == 3:
            iv, e, g, l = pattern(t)
            ex.append((c[1].split('/')[-1], t["probe"], round(t["corr_pair"], 3), round(t["T"] / 1e4, 2), e, g, l,
                       [(round(a, 2), round(b, 2)) for a, b in iv][:8]))
print("three-context probes of the seven-fragment cells: file, probe, r, length (s), early, gap 2.2-3.6 s, late, periods")
for e in ex:
    print("  ", e)
if match:
    r = next(r for r in NAT if r["path"] == match[0][1])
    t = next(t for t in r["tests"] if t["probe"] == match[0][4])
    print("\nmatched example:", match[0][1], "probe", t["probe"], "contexts", t["contexts"], pattern(t))
full = [e for e in ex if e[4] and e[5] and e[6]]
print("three-context probes with the paper's pattern (early, none in 2.2-3.6 s, late):", len(full), "of", len(ex))
late4 = sum(any(b / 1e4 > 4.0 for a, b in t["sig"]) for _, t in nat)
print("natural sound probes with a significant period after 4 s:", late4, "of", len(nat))

**Result: reproduced.** For the matched probe (20051222-ha1-001, probe 4, contexts silence, 2 and 3)
the significant periods run from 0.03 to 1.60 s, none falls between 2.2 and 3.6 s, and they resume
at 3.97 s and last to 4.60 s, the paper's pattern. Of the 96 three-context probes of the
seven-fragment cells, 5 show it; 17 of the 305 natural sound probes have a significant period
after 4 s.

## fig3a-proportion-significant

> Of 305 natural sound probes tested in 39 A1 cells with typically five to eight natural sound
> contexts, 204 (66.9%) showed significant context dependence, and about a quarter (23.7%) of the
> significant events occurred more than 1 s after probe onset.

Readings: silence counted as a context (main) or not; the proportion at t over the probes still
running or over all probes. Noise floor: 1000 shuffles per probe, under the claim's seed and four
more.

In [ ]:
nctx = np.array([len(t["contexts"]) for _, t in nat])
print("contexts per probe (silence counted):", collections.Counter(nctx.tolist()))
for name, tests, sub in (("silence counted", [t for _, t in nat], None), ("silence not counted", [t for _, t in nat_ns], "ns")):
    k = sum(len((t[sub] if sub else t)["sig"]) > 0 for t in tests)
    print(f"\n{name}: significant probes {k} of {len(tests)} ({k / len(tests):.3f})")
    for si in range(5 if sub is None else 1):
        C = A.sig_curve(tests, si, sub)
        for rd in ("running", "all"):
            lf, pk = A.late_fraction(C[rd], C["noise_" + rd])
            print(f"  seed {si}: {rd:7s}: late fraction {lf:.3f}, cumulative peak at {pk:.2f} s, "
                  f"mean noise floor (0-4.72 s) {np.nanmean(C['noise_' + rd][:4720]):.5f}")
# by animal-day
byan = collections.defaultdict(lambda: [0, 0])
for r, t in nat:
    byan[r["animal"]][0] += len(t["sig"]) > 0; byan[r["animal"]][1] += 1
print("\nsignificant/probes by animal-day:", {k: f"{a}/{b}" for k, (a, b) in sorted(byan.items())})
C0 = A.sig_curve([t for _, t in nat], 0)
fig, ax = plt.subplots(figsize=(6, 3))
tm = np.arange(len(C0["running"])) / 1000
ax.plot(tm, C0["running"], "k", label="proportion (running)")
ax.plot(tm, C0["noise_running"], "0.6", label="noise floor")
ax.set_xlim(0, 6); ax.set_ylim(0, 0.35); ax.set_xlabel("time from probe onset (s)"); ax.legend()
print("proportion range over 0-6 s:", np.nanmin(C0['running'][:6000]).round(3), "to", np.nanmax(C0['running'][:6000]).round(3))

**Result: reproduced.** With silence as a context, 186 of 305 probes are significant (0.610),
within 204 +/- 20.4 and 0.669 +/- 0.067. The late fraction is 0.255 when the proportion at t is over
all probes (within 0.237 +/- 0.024) and 0.270 over the probes still running (outside); the
cumulative curve peaks at 4.78 s (paper 4.72 s) and the mean noise floor is 0.00281 (paper
0.0028). The five seeds give 0.254 to 0.255 and 0.270 to 0.271. Without silence as a context,
165 of 305 (0.541) are significant and the late fraction is 0.174 to 0.182, outside.

## fig3a-significance-decay

> The proportion of probes with significant context dependence decays after probe onset as the sum
> of two exponentials (sizes 0.17 and 0.09, decay constants 0.20 and 0.90 s), above a noise floor of
> about 2.8 x 10^-3.

Eq. 44 with the constant fixed at the mean noise floor, the paper's rule for the number of
exponentials. Readings: fit range 0 to 4.72 s or 0 to 6 s; probes running or all; sizes free in
sign, or held non-negative (Eq. 44 names them decay sizes). The noise floors are those of
fig3a-proportion-significant (its seeds).

In [ ]:
def fit_nonneg(t, y, kappa):
    best = None
    for k in (2, 1):
        f = A.fit_exp(t, y, k, kappa)
        if min(f["alpha"]) < 0:
            # refit with sizes bounded at 0
            from scipy import optimize
            p0 = sum(([max(a, 1e-3), tau] for a, tau in zip(f["alpha"], f["tau"])), [])
            lo = sum(([0, 1e-3] for _ in range(k)), []); hi = sum(([np.inf, 100] for _ in range(k)), [])
            res = optimize.least_squares(lambda p: A._model(p, t, k, kappa) - y, p0, bounds=(lo, hi))
            comps = sorted([(res.x[2 * i], res.x[2 * i + 1]) for i in range(k)], key=lambda a: a[1])
            f = dict(k=k, alpha=[a for a, _ in comps], tau=[b for _, b in comps], kappa=kappa)
        a = np.abs(f["alpha"])
        if np.all(a >= a.sum() / 10):
            return f
        best = f
    return best
FIT3A = {}
for si in range(5):
    C = A.sig_curve([t for _, t in nat], si)
    for rd in ("running", "all"):
        for rng in (4.72, 6.0):
            k = int(rng * 1000); tm = np.arange(k) / 1000; y = C[rd][:k]
            kap = float(np.nanmean(C["noise_" + rd][:k]))
            for bound in ("free", "nonneg"):
                f = A.fit_select(tm, y, kappa=kap)[0] if bound == "free" else fit_nonneg(tm, y, kap)
                FIT3A[(si, rd, rng, bound)] = f
                if si == 0 or (rd == "running" and rng == 6.0):
                    print(f"seed {si} {rd:7s} 0-{rng} s {bound:6s}: {f['k']} exp, alpha {np.round(f['alpha'], 3)}, tau {np.round(f['tau'], 3)} s, kappa {kap:.5f}")
def within(f):
    if f["k"] != 2: return False
    a1, a2 = f["alpha"]; t1, t2 = f["tau"]
    return abs(a1 - .17) <= .017 and abs(t1 - .2) <= .02 and abs(a2 - .09) <= .009 and abs(t2 - .9) <= .09
print("\nreadings with two exponentials and all values within tolerance:", [k for k, f in FIT3A.items() if within(f)])
print("readings with two exponentials and slow constant 0.5-2 s:", [k for k, f in FIT3A.items() if f['k'] == 2 and 0.5 <= f['tau'][1] <= 2])

**Result: partially reproduced.** No reading puts every value within tolerance. Fitted from 0 to 6 s
over the running probes, two exponentials are selected with sizes 0.159 and 0.072 and constants
0.26 and 1.05 s (paper 0.17, 0.09, 0.20 and 0.90 s); over all probes, 0.151 and 0.081 with 0.25
and 0.98 s. The slow constant is between 0.5 and 2 s under every reading that selects two
decaying exponentials and under all five seeds. Fitted from 0 to 4.72 s with free signs, the fast
component is a rise (size -0.172, 15 ms) at the onset.

## fig3b-fractional-power-decay

> The context-dependent fraction of the stimulus-related response power, normalized by the smoothed
> stimulus-related power, decays from about 0.49 with a time constant of about 1.04 s.

Q(t) of Eq. 20, one exponential with constant 0. Readings: silence counted or not; probes running
or all; fit range 0 to 4 s or 0 to 6 s.

In [ ]:
FIT3B = {}
for name, tests, sub in (("silence counted", [t for _, t in nat], None), ("silence not counted", [t for _, t in nat_ns], "ns")):
    PC = A.power_curves(tests, sub)
    for rd in ("running", "all"):
        for rng in (4.0, 6.0):
            k = int(rng * 1000); tm = np.arange(k) / 1000
            f = A.fit_exp(tm, PC[rd]["Q"][:k], 1, kappa=0.0)
            FIT3B[(name, rd, rng)] = f
            print(f"{name:19s} {rd:7s} 0-{rng} s: alpha {f['alpha'][0]:.3f}, tau {f['tau'][0]:.3f} s; "
                  f"Q at 0.1, 1, 2, 4 s: {np.round(PC[rd]['Q'][[100, 1000, 2000, 3999]], 3)}")
    if sub is None:
        PC0 = PC
print("\nreadings with both within tolerance:", [k for k, f in FIT3B.items() if abs(f['alpha'][0] - .49) <= .049 and abs(f['tau'][0] - 1.04) <= .104])
fig, ax = plt.subplots(figsize=(6, 3))
tm = np.arange(len(PC0["running"]["Q"])) / 1000
ax.plot(tm, PC0["running"]["Q"], "k"); ax.set_xlim(0, 6); ax.set_ylim(-0.2, 1); ax.set_xlabel("time from probe onset (s)"); ax.set_ylabel("Q(t)")
q = PC0["running"]["Q"][:6000]
print("Q range over 0-6 s:", np.nanmin(q).round(3), "to", np.nanmax(q).round(3), "; points outside the axes:", int(((q < -0.2) | (q > 1)).sum()))

**Result: partially reproduced.** The size is within tolerance under every reading (0.445 to 0.468
against 0.49 +/- 0.049) and the decay constant is longer than 1.04 s under every reading (1.29 to
1.78 s, tolerance 0.104). Q(t) is 0.626 at 0.1 s and 0.191 at 1 s, the value of the paper's curve
there (0.49 exp(-1/1.04) = 0.188), and 0.049 at 2 s; it rises to 0.13 at 4 s, which lengthens the
fitted constant.

## timescale-longer-than-membrane

> The trial-to-trial noise autocorrelation and the autocorrelation of spontaneous activity decay
> within about 100 ms, much faster than the second-long timescale of context dependence in A1.

Noise: residual of each trial from its context mean (Eq. 14), autocorrelation normalised at lag 0
(Eq. 45), averaged over trials and probes. Spontaneous: the intersequence silences, whole or
without the first second. Lags at 1 ms.

In [ ]:
def lag_below(ac, level):
    i = np.flatnonzero(ac < level)
    return float(i[0]) if len(i) else np.nan  # ms
acn = np.mean([t["acf_noise"] for _, t in nat], axis=0)
acs = np.mean([r["acf_spont"] for r in NAT if r["acf_spont"] is not None], axis=0)
acl = np.mean([r["acf_spont_late"] for r in NAT if r["acf_spont_late"] is not None], axis=0)
for name, ac in (("noise", acn), ("spontaneous, whole silence", acs), ("spontaneous, after 1 s", acl)):
    print(f"{name:28s}: lag below 1/e {lag_below(ac, np.exp(-1)):.0f} ms, below 0.1 {lag_below(ac, 0.1):.0f} ms; "
          f"value at 10, 50, 100, 500 ms {np.round(ac[[10, 50, 100, 500]], 3)}")
per_file = [lag_below(r["acf_spont"], np.exp(-1)) for r in NAT if r["acf_spont"] is not None]
print("spontaneous 1/e lag per file (ms): median", np.median(per_file), "range", np.min(per_file), "to", np.max(per_file))
per_probe = [lag_below(t["acf_noise"], np.exp(-1)) for _, t in nat]
print("noise 1/e lag per probe (ms): median", np.median(per_probe), "range", np.min(per_probe), "to", np.max(per_probe))

**Result: reproduced.** The mean noise autocorrelation falls below 1/e at 71 ms and below 0.1 at 115
ms; the spontaneous autocorrelation at 77 and 121 ms over whole silences, and at 75 and 119 ms
without their first second. Both 1/e lags are under 150 ms and under a third of 1.04 s.

## fig4-predictability-upper-bound

> Without knowledge of the context, no model can capture more than about half (0.51) of the
> predictable response power with a window of about 100 ms; the upper bound approaches one only with
> a window of seconds (time constant 1.04 s).

Eq. 43, one exponential with constant 1. Readings: denominator smoothed over [0, 2t] (main) or
not; silence counted or not; probes running or all; fit range 0 to 4 s or 0 to 6 s.

In [ ]:
FIT4 = {}
for name, tests, sub in (("silence counted", [t for _, t in nat], None), ("silence not counted", [t for _, t in nat_ns], "ns")):
    PC = A.power_curves(tests, sub)
    for rd in ("running", "all"):
        for key in ("U", "U_raw"):
            for rng in (4.0, 6.0):
                k = int(rng * 1000); tm = np.arange(k) / 1000
                f = A.fit_exp(tm, PC[rd][key][:k], 1, kappa=1.0)
                FIT4[(name, rd, key, rng)] = f
                print(f"{name:19s} {rd:7s} {'smoothed' if key == 'U' else 'raw     '} 0-{rng} s: 1 + alpha {1 + f['alpha'][0]:.3f}, tau {f['tau'][0]:.3f} s; "
                      f"ratio at 0.1 s {PC[rd][key][100]:.3f}")
print("\nreadings with both within tolerance:", [k for k, f in FIT4.items() if abs(1 + f['alpha'][0] - .51) <= .051 and abs(f['tau'][0] - 1.04) <= .104])

**Result: partially reproduced.** No reading puts both values within tolerance. With the denominator
smoothed as the Methods say, 1 + alpha is 0.43 to 0.47 over the running probes (0.51 +/- 0.051) and
tau 1.86 to 2.81 s; unsmoothed, 1 + alpha is 0.50 to 0.54, within tolerance, and tau 1.46 to 1.92
s. The bound is below 1 and approaches it over seconds under every reading, with a constant longer
than the paper's 1.04 s.

## fig8a-proportion-by-property

> When only one property of synthetic conditioning stimuli varied, context dependence was found in
> 77 of 95 probes (81.1%, 31 cells) for amplitude, 73 of 110 (35 cells) for frequency, 62 of 96
> (64.6%, 27 cells) for AM, 47 of 82 (57.3%, 25 cells) for FM and 59 of 137 (43.1%, 27 cells) for
> higher-order properties.

Main reading: each probe with each set of conditioning stimuli that differ in one property is a
test. Variant: each probe counted once per property, significant when any of its sets is.

In [ ]:
EXP8 = {"amplitude": (77, 95), "frequency": (73, 110), "AM": (62, 96), "FM": (47, 82), "higher-order": (59, 137)}
res8 = {}
for p in A.PROPERTIES:
    T = [(r, t) for r, t in syn if t["prop"] == p]
    k = sum(len(t["sig"]) > 0 for _, t in T)
    once = collections.defaultdict(bool)
    for r, t in T:
        once[(r["path"], t["probe"])] |= len(t["sig"]) > 0
    res8[p] = (k, len(T), len({r['cell'] for r, _ in T}), sum(once.values()), len(once))
    e = EXP8[p]
    print(f"{p:12s}: {k} of {len(T)} ({k / len(T):.3f}; {res8[p][2]} cells)  paper {e[0]} of {e[1]} ({e[0] / e[1]:.3f}); "
          f"within tolerance {abs(k - e[0]) <= 0.1 * e[0]};  once per probe: {sum(once.values())} of {len(once)} ({sum(once.values()) / len(once):.3f})")
pr = {p: res8[p][0] / res8[p][1] for p in A.PROPERTIES}
print("order amplitude > frequency:", pr["amplitude"] > pr["frequency"], "; AM > FM > higher-order:", pr["AM"] > pr["FM"] > pr["higher-order"])
pr1 = {p: res8[p][3] / res8[p][4] for p in A.PROPERTIES}
print("once-per-probe order amplitude > frequency:", pr1["amplitude"] > pr1["frequency"], "; AM > FM > higher-order:", pr1["AM"] > pr1["FM"] > pr1["higher-order"])

**Result: partially reproduced.** Significant tests: amplitude 61 of 104, frequency 70 of 120, AM 44
of 96, FM 29 of 82 and higher-order 109 of 354; only frequency is within tolerance of the paper's
count (73). The proportions keep the paper's order, amplitude (0.587) above frequency (0.583) and
AM (0.458) above FM (0.354) above higher-order (0.308), as they do when each probe is counted once
(0.611, 0.564, 0.718, 0.640, 0.467). The AM and FM denominators equal the paper's; the amplitude,
frequency and higher-order ones do not.

## fig8b-decay-by-property

> The normalized context-dependent power decays slowly for amplitude and frequency changes (time
> constants 0.66 and 0.97 s) and fast for AM, FM and higher-order changes (0.11, 0.15 and 0.20 s).

Q(t) per property over the tests of fig8a-proportion-by-property (probes running), one
exponential with constant 0, fitted from 0 to 4 s; 0 to the shortest probe length as a variant.

In [ ]:
EXP8B = {"amplitude": (0.66, 0.57), "frequency": (0.97, 0.54), "AM": (0.11, 0.86), "FM": (0.15, 0.64), "higher-order": (0.20, 0.52)}
FIT8B = {}
PC8 = {}
for p in A.PROPERTIES:
    T = [t for r, t in syn if t["prop"] == p]
    PC = A.power_curves(T)
    PC8[p] = PC
    short = min(t["T"] for t in T) / 1e4
    for rng in sorted({4.0, round(short, 2)}):
        k = int(rng * 1000); tm = np.arange(k) / 1000
        f = A.fit_exp(tm, PC["running"]["Q"][:k], 1, kappa=0.0)
        FIT8B[(p, round(rng, 2))] = f
        print(f"{p:12s} 0-{rng:.2f} s: alpha {f['alpha'][0]:.3f}, tau {f['tau'][0]:.3f} s (paper {EXP8B[p][1]}, {EXP8B[p][0]} s); "
              f"tau within tolerance {abs(f['tau'][0] - EXP8B[p][0]) <= 0.1 * EXP8B[p][0]}")
t4 = {p: FIT8B[(p, 4.0)]["tau"][0] for p in A.PROPERTIES}
print("partial pattern (amplitude, frequency > 0.4 s; AM, FM, higher-order < 0.3 s):",
      t4["amplitude"] > .4 and t4["frequency"] > .4 and max(t4["AM"], t4["FM"], t4["higher-order"]) < .3)

**Result: partially reproduced.** Fitted from 0 to 4 s, the decay constants are 1.35 s (amplitude),
0.84 s (frequency), 0.11 s (AM), 0.28 s (FM) and 0.23 s (higher-order); only AM is within
tolerance (0.11 s). The pattern of the partial rule holds: amplitude and frequency above 0.4 s, the
other three below 0.3 s. The sizes are 0.37, 0.48, 0.85, 0.21 and 0.82 against 0.57, 0.54, 0.86,
0.64 and 0.52.

## fig8a-significance-fits

> The proportion of probes with significant context dependence decays fastest and is smallest for
> higher-order changes, with fitted (size, time constant) of (0.23, 0.25 s) for amplitude, (0.19,
> 0.27 s) for frequency, (0.13, 0.17 s) for AM, (0.07, 0.15 s) for FM and (0.04, 0.13 s) for
> higher-order properties.

One exponential with the constant at the mean noise floor of each property (probes running),
fitted from 0 to 4 s, under the claim's seed and four more; sizes free in sign or non-negative.

In [ ]:
EXP8A = {"amplitude": (0.23, 0.25, 2.8e-3), "frequency": (0.19, 0.27, 2.3e-3), "AM": (0.13, 0.17, 2.3e-3),
         "FM": (0.07, 0.15, 2.1e-3), "higher-order": (0.04, 0.13, 3.6e-3)}
FIT8A = {}
for si in range(5):
    for p in A.PROPERTIES:
        T = [t for r, t in syn if t["prop"] == p]
        C = A.sig_curve(T, si)
        k = 4000; tm = np.arange(k) / 1000
        kap = float(np.nanmean(C["noise_running"][:k]))
        for bound in ("free", "nonneg"):
            if bound == "free":
                f = A.fit_exp(tm, C["running"][:k], 1, kappa=kap)
            else:
                from scipy import optimize
                f0 = A.fit_exp(tm, C["running"][:k], 1, kappa=kap)
                res = optimize.least_squares(lambda q: A._model(q, tm, 1, kap) - C["running"][:k],
                                             [max(f0["alpha"][0], 1e-3), f0["tau"][0]], bounds=([0, 1e-3], [np.inf, 100]))
                f = dict(alpha=[res.x[0]], tau=[res.x[1]], kappa=kap)
            FIT8A[(si, p, bound)] = f
            if si == 0 or bound == "free":
                e = EXP8A[p]
                ok = abs(f["alpha"][0] - e[0]) <= 0.1 * e[0] and abs(f["tau"][0] - e[1]) <= 0.1 * e[1]
                print(f"seed {si} {p:12s} {bound:6s}: alpha {f['alpha'][0]:.3f}, tau {f['tau'][0]:.3f} s, noise floor {kap:.5f} "
                      f"(paper {e}); both within tolerance {ok}; p(0-10 ms) {np.nanmean(C['running'][:10]):.3f}")
for bound in ("free", "nonneg"):
    sizes = [FIT8A[(0, p, bound)]["alpha"][0] for p in A.PROPERTIES]
    print(bound, "sizes in the printed order (amplitude > frequency > AM > FM > higher-order):", all(np.diff(sizes) < 0), np.round(sizes, 3))

**Result: not reproduced.** No size is within tolerance under any seed: 0.121, 0.150, 0.067, 0.013
and 0.035 against 0.23, 0.19, 0.13, 0.07 and 0.04, with constants 0.27, 0.29, 0.20, 0.14 and 0.11
s. The sizes are not in the printed order: frequency is above amplitude and higher-order above FM.
The fitted noise floors (0.0033, 0.0023, 0.0021, 0.0021, 0.0023) are close to the legend's except
for higher-order (0.0036). Bounding the sizes at 0 changes nothing.

## fig8c-relative-effects

> Changes in amplitude and frequency of the conditioning stimuli produced context effects as large
> and long-lasting as natural sound contexts, while AM, FM and higher-order changes produced
> substantially smaller and shorter effects.

Area under Q(t) from 0 to 4 s for each property over the area for natural sounds (silence counted,
probes running). 95% intervals from 200 resamplings of 1000 probes drawn with replacement, within
each property and within the natural sounds (Methods, Eq. 21 and the paragraph after it). The
reading where trials are resampled was not run.

In [ ]:
def area(tests, idx=None):
    if idx is not None:
        tests = [tests[i] for i in idx]
    PC = A.power_curves(tests)
    return float(np.nansum(PC["running"]["Q"][:4000]) / 1000)
nat_tests = [t for _, t in nat]
a_nat = area(nat_tests)
print("area 0-4 s, natural sounds:", round(a_nat, 3))
REL = {}
for si, s in enumerate(five("fig8c-relative-effects")):
    rng = np.random.default_rng(s)
    boot_nat = []
    for b in range(200):
        boot_nat.append(area(nat_tests, rng.integers(0, len(nat_tests), 1000)))
    for p in A.PROPERTIES:
        T = [t for r, t in syn if t["prop"] == p]
        point = area(T) / a_nat
        bs = np.array([area(T, rng.integers(0, len(T), 1000)) / boot_nat[b] for b in range(200)])
        REL[(si, p)] = (point, *np.percentile(bs, [2.5, 97.5]))
    if si == 0 or True:
        print(f"seed {si}:", {p: tuple(np.round(REL[(si, p)], 3)) for p in A.PROPERTIES})
def pattern8c(si):
    r = {p: REL[(si, p)] for p in A.PROPERTIES}
    hi = all(r[p][2] >= 1 for p in ("amplitude", "frequency"))
    lo_bound = min(r["amplitude"][1], r["frequency"][1])
    low = all(r[p][2] < 1 and r[p][2] < lo_bound for p in ("AM", "FM", "higher-order"))
    pts = min(r["amplitude"][0], r["frequency"][0]) > max(r[p][0] for p in ("AM", "FM", "higher-order"))
    return hi, low, pts
for si in range(5):
    print(f"seed {si}: amplitude and frequency intervals reach 1: {pattern8c(si)[0]}; AM, FM, higher-order below 1 and below "
          f"their lower bounds: {pattern8c(si)[1]}; point estimates in that order: {pattern8c(si)[2]}")

**Result: partially reproduced.** Relative to natural sounds, the areas are 0.67 (amplitude), 0.63
(frequency), 0.15 (AM), 0.13 (FM) and 0.37 (higher-order). The intervals for AM, FM and
higher-order lie below 1 and below the lower bounds of amplitude (0.58) and frequency (0.55), but
the amplitude and frequency intervals (upper bounds 0.78 and 0.72) do not reach 1. The point
estimates fall in the claimed pattern under all five seeds.

## fig8-natural-synthetic-probes-alike

> Natural and synthetic probes gave no difference in the context effects of the synthetic
> conditioning stimuli, so they were combined.

Fisher's exact test per property of the proportion of significant tests, natural against
synthetic probes, at 0.05 with Bonferroni over the properties tested with both.

In [ ]:
rows = []
for p in A.PROPERTIES:
    T = [t for r, t in syn if t["prop"] == p]
    a = sum(len(t["sig"]) > 0 for t in T if t["natural"]); n1 = sum(t["natural"] for t in T)
    b = sum(len(t["sig"]) > 0 for t in T if not t["natural"]); n2 = len(T) - n1
    if n1 and n2:
        o, pv = stats.fisher_exact([[a, n1 - a], [b, n2 - b]])
        rows.append((p, a, n1, b, n2, o, pv))
    else:
        print(f"{p}: only {'natural' if n1 else 'synthetic'} probes ({n1 + n2})")
m = len(rows)
for p, a, n1, b, n2, o, pv in rows:
    print(f"{p:12s}: natural {a}/{n1} ({a / n1:.3f}), synthetic {b}/{n2} ({b / n2:.3f}); odds ratio {o:.2f}, "
          f"p = {pv:.4f}, Bonferroni p = {min(1, pv * m):.4f}")
print("properties with a difference after Bonferroni:", [r[0] for r in rows if r[-1] * m < 0.05])

**Result: reproduced, resting on the test fixed by the claims file.** No property differs between
natural and synthetic probes: amplitude 0.556 against 0.620 (odds ratio 0.77, p = 0.55), frequency
0.554 against 0.609 (0.79, p = 0.58), AM 0.439 against 0.571 (0.59, p = 0.40), higher-order 0.341
against 0.279 (1.34, p = 0.21), Bonferroni over four. FM was tested with natural probes only.

## fig9-mgb-fast-decay

> In the auditory thalamus (MGB), context dependence of spike responses decayed within a short time
> (time constant about 80 ms; 93 probes in 14 cells).

PSTHs in 100-ms bins from probe onset, SD over contexts (n - 1, Eq. 42), averaged over the probes
still having the bin. One exponential with a free constant over bins 0 to 60, the bin's time at its
start or its centre; spikes by the authors' detector and with half its threshold.

In [ ]:
FIT9 = {}
for key in ("code", "half"):
    pr = [(r, p) for r in MGB if (r["rep"] or 0) >= 10 for p in r[f"probes_{key}"]]
    sd, _, n = A.pop_average([p["sd"] for _, p in pr])
    k = min(61, len(sd)); tb = np.arange(k) * 0.1
    print(f"detector {key}: {len(pr)} probes; mean SD in the first bins {np.round(sd[:8], 2)}; mean after 0.5 s {np.nanmean(sd[5:k]):.2f} spike/s")
    for off in (0.0, 0.05):
        f = A.fit_exp(tb + off, sd[:k], 1)
        FIT9[(key, off)] = (f, float(np.nanmean(sd[5:k])))
        print(f"   bin time at {'start ' if off == 0 else 'centre'}: alpha {f['alpha'][0]:.2f} spike/s, tau {f['tau'][0] * 1000:.0f} ms, kappa {f['kappa']:.2f} spike/s")
    bycell = collections.defaultdict(list)
    for r, p in pr:
        bycell[r["cell"]].append(p["sd"][0] / max(np.mean(p["sd"][1:]), 1e-9))
    print("   per cell, median ratio of the first bin's SD to later bins':", {c: round(float(np.median(v)), 1) for c, v in bycell.items()})

**Result: partially reproduced.** The mean SD of PSTHs is 6.45 spike/s in the first bin and 2.04 in
the second; a single exponential gives tau 55 ms (61 ms with the lower threshold), below 0.1 s,
with sizes 5.2 or 12.9 spike/s (bin start or centre; paper 15.0). The plateau after 0.5 s is 1.27
spike/s (1.38), more than 25% below the paper's constant of 2.0, so the second condition fails.

## fig9-example-first-bin

> In an example MGB unit, the response to a probe differed between contexts only in the first
> 100-ms bin of the PSTH.

Among MGB probes of the Figure 9A design (the 7-long-stimuli sequence, three contexts: silence and
two fragments), those whose first-bin SD over contexts is at least twice every later bin's; the
fraction over all 93 probes too.

In [ ]:
for key in ("code", "half"):
    pr = [(r, p) for r in MGB if (r["rep"] or 0) >= 10 for p in r[f"probes_{key}"]]
    good = [(r, p) for r, p in pr if len(p["sd"]) > 1 and p["sd"][0] >= 2 * np.max(p["sd"][1:]) and p["sd"][0] > 0]
    des = [(r, p) for r, p in pr if len(r["seqs"]) == 7 and len(p["contexts"]) == 3]
    gd = [(r, p) for r, p in des if (r, p) in good or any(p is q for _, q in good)]
    print(f"detector {key}: probes with first bin >= 2x every later bin: {len(good)} of {len(pr)} ({len(good) / len(pr):.3f}); "
          f"of the Figure 9A design (7 fragments, 3 contexts): {len(gd)} of {len(des)}")
    for r, p in gd[:5]:
        print("    ", r["path"].split("/")[-1], "probe", p["probe"], "contexts", p["contexts"], "SD:", np.round(p["sd"][:6], 1))

**Result: reproduced.** Of the 12 probes with the Figure 9A design (all in 20071010-ha2-003), 4
have a first-bin SD at least twice every later bin's (for example 31.4 spike/s against at most 7.2),
under both detectors; over all 93 probes, 35 (code detector) or 45 do.

## figs1-silence-probe

> A subset of cells showed significant context dependence soon after the onset of the intersequence
> silences, but the response power attributable to the preceding stimulus alone faded within about
> 1 s after the stimulus ended.

Silence probes: the first 2.5 s of the interval after each sequence (CxtDep.m's blank probe), with
the sequence's last fragment as the context. Significance as for any probe; power: stimulus-related
power (Eq. 17, main) and context-dependent power (Eq. 19, variant), population averages over silence
probes, against their value in the first 10 ms.

In [ ]:
bl = [(r, r["blank"]) for r in NAT if r["blank"] is not None and (r["rep"] or 0) >= 4]
early = [(r["cell"], b["sig"][0][0] / 1e4) for r, b in bl if b["sig"] and b["sig"][0][0] / 1e4 <= 0.5]
anyp = [r["cell"] for r, b in bl if b["sig"]]
print(f"silence probes: {len(bl)} (files) in {len({r['cell'] for r, _ in bl})} cells; with any significant period: {len(anyp)}; "
      f"starting within 0.5 s: {len(early)} in {len({c for c, _ in early})} cells")
print("silence probe lengths (s):", sorted({round(b['T'] / 1e4, 2) for _, b in bl}), "; contexts per probe:", collections.Counter(len(b['contexts']) for _, b in bl))
pred, _, _ = A.pop_average([b["pred"] for _, b in bl]); inv, _, _ = A.pop_average([b["inv"] for _, b in bl])
for name, y in (("stimulus-related (Eq. 17)", pred), ("context-dependent (Eq. 19)", pred - inv)):
    y0 = np.nanmean(y[:10])
    below = np.flatnonzero(y[10:] < 0.1 * y0)
    tb = (below[0] + 10) / 1000 if len(below) else np.nan
    print(f"{name}: value at 0-10 ms {y0:.2f} mV^2; at 0.5, 1.0, 1.1, 2.0 s: {np.round(y[[500, 1000, 1100, 2000]], 2)}; "
          f"first below 10% of onset at {tb:.3f} s; mean over 1.1-2.5 s relative to onset {np.nanmean(y[1100:]) / y0:.3f}")

**Result: reproduced.** Of 44 silence probes, 24 in 21 cells have a significant period starting
within 0.5 s of the silence onset. The stimulus-related power over silence probes falls from 4.86
mV^2 in the first 10 ms below 10% of that at 0.72 s, and the context-dependent power at 0.45 s,
both before 1.1 s.

## Summary

| Claim | Outcome |
|---|---|
| methods-recording-counts | partially reproduced |
| a1-mgb-firing-rates | partially reproduced |
| fig2-example-reliability | reproduced |
| fig2-example-long-context | reproduced |
| fig3a-proportion-significant | reproduced |
| fig3a-significance-decay | partially reproduced |
| fig3b-fractional-power-decay | partially reproduced |
| timescale-longer-than-membrane | reproduced |
| fig4-predictability-upper-bound | partially reproduced |
| fig4-strf-performance-low | blocked (not run in the time of this attempt) |
| fig5-example-conductances | blocked (data missing: no voltage clamp) |
| fig6-conductance-decay | blocked (data missing: no voltage clamp) |
| fig8a-proportion-by-property | partially reproduced |
| fig8b-decay-by-property | partially reproduced |
| fig8a-significance-fits | not reproduced |
| fig8c-relative-effects | partially reproduced |
| fig8-natural-synthetic-probes-alike | reproduced (assumed test) |
| fig9-mgb-fast-decay | partially reproduced |
| fig9-example-first-bin | reproduced |
| fig7-example-frequency-context | blocked (no recording matches) |
| figs1-silence-probe | reproduced |

A failure to reproduce here may be a failure of this attempt, not of the paper: see the
reasoning and the differences from the paper's method.

How the analysis differs from the paper's. Details the Methods leave open are taken from the
authors' code in the data set (CxtDep.m): silence as a context, trials per context, the
significance run rule, the centring of traces for the power measure, the MGB spike detector. The
A1 spike detector is assumed (the authors extracted whole-cell spikes by hand). The property sets
of Figure 8 are formed here from the stimulus parameters, and their totals differ from Table 1 for
amplitude, frequency and higher-order properties. Curves are fitted at 1-ms resolution. The
Figure 8C intervals resample probes only. The STRF claim was not run.

What was found about the data set. Asari's natural sound and thalamic recordings match the paper's
counts exactly (305 probes in 39 cells; 93 probes in 14 cells). The data set holds no
voltage-clamp recordings, though the paper analyses 14 voltage-clamped cells. It holds 145
current-clamp A1 cells where the paper recorded 194, and 14 MGB cells where it recorded 16. In two
files (20051107-ha1-001-003 and -004) the trigger descriptions number the sequences 2, 4, 6, 8
while the notes number them 1 to 4; the sequences are matched by content. The analysis code in
`asari/Tools` documents details the paper leaves open. The trial tables share the traces' clock.

In [ ]:
print("total time (min):", round((time.time() - T0) / 60, 1))
print("peak memory (MB):", resource.getrusage(resource.RUSAGE_SELF).ru_maxrss // 1024)